# ?? SignalScope: Master Dual-Stream ConvNeXt-Tiny + SRM Forensic Model Trainer
**Smart India Hackathon (SIH 2026) | Problem Statement 2**
Domain: **AI Media Forensics / Trust & Safety**

### ?? Primary Objective: Generalization to Unseen AI Generators
- **Semantic Stream**: ConvNeXt-Tiny (Pretrained)
- **Forensic Stream**: Spatial Rich Model (SRM) High-Pass Noise Residuals
- **Training Strategy**: Mixed-Precision (AMP) Two-Phase Differential Fine-Tuning
- **Evaluation Split**: Held-Out Midjourney & VQDM (Zero exposure during training)

> ?? **IMPORTANT**: Pehle menu mein `Runtime` -> `Change runtime type` -> select **T4 GPU** karein.

## 1. Verify GPU Environment

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: GPU not detected! Please set Runtime -> Change runtime type -> T4 GPU.")

## 2. Mount Google Drive & Install Required Libraries

In [ ]:
from google.colab import drive
# Force remount so you can select vishvjani87@gmail.com
drive.mount('/content/drive', force_remount=True)

!pip install -q timm pyyaml matplotlib scikit-learn

## 3. Clone Repository & Setup Working Directory

In [ ]:
import os
import sys

if not os.path.exists('/content/sih_1'):
    !git clone https://github.com/vishvjani/sih_1.git /content/sih_1
else:
    %cd /content/sih_1
    !git pull origin main

%cd /content/sih_1
for p in ['/content/sih_1/model_engine', '/content/sih_1']:
    if p not in sys.path:
        sys.path.insert(0, p)
print('Current Working Directory:', os.getcwd())

## 4. Dataset Setup
Agar aapne GenImage ki `.zip` file apne Google Drive mein download ki hai, toh niche diye cell se unzip karein. Agar koi zip nahi hai, toh yeh cell automatically sample dataset taiyar kar dega taaki training turant chal sake.

In [ ]:
import os
import shutil
from pathlib import Path
from PIL import Image
import numpy as np

data_root = Path('/content/data/GenImage')
data_root.mkdir(parents=True, exist_ok=True)
drive_path = Path('/content/drive/MyDrive')

# 1. Check if 'genimage' folder exists directly in vishvjani87@gmail.com Drive (from shortcut)
genimage_direct = drive_path / 'genimage'
if not genimage_direct.exists():
    genimage_direct = drive_path / 'GenImage'

if genimage_direct.exists():
    print(f"✅ Found official GenImage folder directly in Drive: {genimage_direct}")
    data_root = genimage_direct
else:
    # 2. Case-insensitive search for dataset zips in Google Drive
    found_zips = []
    dataset_keywords = ['genimage', 'diffusion', 'imagenet', 'sih', 'dataset', 'midjourney', 'glide', 'biggan', 'wukong', 'vqdm', 'adm']
    if drive_path.exists():
        for f in drive_path.rglob('*.zip'):
            name_lower = f.name.lower()
            if any(keyword in name_lower for keyword in dataset_keywords):
                found_zips.append(f)

    if found_zips:
        print(f"Found {len(found_zips)} dataset zip file(s) in Drive:")
        for z in found_zips:
            print(f"  Extracting: {z.name} ...")
            !unzip -qo "{z}" -d /content/data/GenImage/
        print("All zips extracted successfully!")
    else:
        # 3. Check for any individual generator folders in Drive
        drive_gen_dirs = []
        if drive_path.exists():
            for p in drive_path.glob('*'):
                if p.is_dir() and any(k in p.name.lower() for k in dataset_keywords):
                    drive_gen_dirs.append(p)
        
        if drive_gen_dirs:
            print(f"Found unzipped dataset folder(s) directly in Drive: {[p.name for p in drive_gen_dirs]}")
            data_root = drive_gen_dirs[0]
        else:
            print("No Drive zip or folder found. Generating structured sample dataset for pipeline verification...")
            for gen in ['stable_diffusion_v1_4', 'glide', 'wukong', 'biggan', 'stable_diffusion_v1_5', 'midjourney', 'vqdm']:
                for split in ['train', 'val']:
                    (data_root / gen / split / 'nature').mkdir(parents=True, exist_ok=True)
                    (data_root / gen / split / 'ai').mkdir(parents=True, exist_ok=True)
                    for i in range(25):
                        real_img = Image.fromarray(np.uint8(np.random.rand(256, 256, 3) * 255))
                        ai_img = Image.fromarray(np.uint8(np.random.rand(256, 256, 3) * 255))
                        real_img.save(data_root / gen / split / 'nature' / f'real_{i}.jpg')
                        ai_img.save(data_root / gen / split / 'ai' / f'ai_{i}.jpg')

# 4. Auto-detect real root if nested directories were created
possible_roots = [data_root] + list(data_root.glob('*')) + list(data_root.glob('*/*')) + list(data_root.glob('*/*/*'))
for cand in possible_roots:
    if cand.is_dir():
        has_gens = any((cand / g).exists() for g in ['stable_diffusion_v1_4', 'Stable Diffusion V1.4', 'midjourney', 'Midjourney', 'biggan', 'BigGAN', 'glide', 'GLIDE', 'wukong', 'Wukong', 'vqdm', 'VQDM', 'imagenet_ai'])
        has_splits = (cand / 'train').exists() or (cand / 'nature').exists()
        if has_gens or has_splits:
            data_root = cand
            break

print(f"Verified active dataset root: {data_root}")


## 5. Phase 0: Dataset Audit & Anti-Leakage Manifest Creation

In [ ]:
from pathlib import Path
import sys
for p in ['/content/sih_1/model_engine', '/content/sih_1']:
    if p not in sys.path:
        sys.path.insert(0, p)

from src.data.audit import DatasetAuditor
from src.data.split import GeneratorSplitter

output_dir = Path('/content/sih_1/manifests')
output_dir.mkdir(parents=True, exist_ok=True)

auditor = DatasetAuditor(data_root)
gen_folders = [p for p in data_root.iterdir() if p.is_dir()]
unique_reals_dict, dupes = auditor.find_real_image_duplicates(gen_folders)
unique_reals = list(unique_reals_dict.values())

# Collect AI paths by generator
ai_by_gen = {}
for gen_dir in gen_folders:
    ai_imgs = list((gen_dir / 'train' / 'ai').glob('*.*')) + list((gen_dir / 'val' / 'ai').glob('*.*'))
    if len(ai_imgs) > 0:
        ai_by_gen[gen_dir.name] = ai_imgs

print(f"Found {len(unique_reals)} unique real images and {sum(len(v) for v in ai_by_gen.values())} AI images across {len(ai_by_gen)} generators.")

splitter = GeneratorSplitter(seed=42)
manifests = splitter.create_100k_manifests(unique_reals, ai_by_gen, output_dir)
print("Manifests created successfully:")
for k, v in manifests.items():
    print(f"  {k}: {v}")

## 6. Initialize Dual-Stream Architecture (ConvNeXt-Tiny + SRM)

In [ ]:
import torch
from src.models.network import DualStreamSignalScope

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = DualStreamSignalScope(pretrained=True, dropout_rate=0.3, use_srm_stream=True).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Initialized on {device}")
print(f"Total Parameters: {total_params / 1e6:.2f}M | Initial Trainable: {trainable_params / 1e6:.2f}M")

## 7. Two-Phase Differential Training (Mixed Precision AMP)
- **Phase 1**: Backbone Frozen, Classifier Warmup
- **Phase 2**: Stage 3 & 4 Differential Fine-Tuning
- Automatically saves `signalscope_final_calibrated.pth` in `/content/sih_1/checkpoints/`

In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader
from src.preprocessing.transforms import get_training_transforms, get_inference_transforms
from src.data.dataset import GenImageDataset
from src.training.trainer import SignalScopeTrainer

manifest_dir = Path('/content/sih_1/manifests')
train_manifest = manifest_dir / 'train_manifest.json'
val_manifest = manifest_dir / 'val_manifest.json'
test_manifest = manifest_dir / 'test_unseen_manifest.json'

train_ds = GenImageDataset(str(train_manifest), transform=get_training_transforms(256))
val_ds = GenImageDataset(str(val_manifest), transform=get_inference_transforms(256))
test_ds = GenImageDataset(str(test_manifest), transform=get_inference_transforms(256))

batch_size = 32 if torch.cuda.is_available() else 4
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Ready to train: {len(train_ds)} train samples, {len(val_ds)} val samples, {len(test_ds)} unseen test samples.")

checkpoint_dir = Path('/content/sih_1/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

trainer = SignalScopeTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_unseen_loader=test_loader,
    device=device,
    checkpoint_dir=str(checkpoint_dir)
)

# Phase 1: 3 epochs warmup, Phase 2: 7 epochs fine-tuning
training_history = trainer.train(phase1_epochs=3, phase2_epochs=7)
print('Training completed! History:', training_history)

## 8. Benchmark Evaluation on Unseen Generators
Evaluates model on held-out Midjourney and VQDM images.

In [ ]:
metrics, logits, labels = trainer.evaluate(test_loader)
print('\n' + '=' * 45)
print('UNSEEN GENERATOR EVALUATION BENCHMARK')
print('=' * 45)
print(f"Overall ROC-AUC:          {metrics.get('overall_roc_auc', 0.0):.4f}")
print(f"Unseen-Gen ROC-AUC:       {metrics.get('unseen_generator_roc_auc', 0.0):.4f}")
print(f"Macro-F1 Score:           {metrics.get('macro_f1', 0.0):.4f}")
print(f"False Positive Rate (FPR): {metrics.get('false_positive_rate', 0.0):.4f}")
print('Confusion Matrix:', metrics.get('confusion_matrix', {}))

## 9. Grad-CAM Localized Visual Explanations
Generates visual attribution heatmap for an unseen test sample.

In [ ]:
from src.explainability.gradcam import GradCAM
import matplotlib.pyplot as plt

gradcam = GradCAM(model)
sample_batch = next(iter(test_loader))
sample_img_t = sample_batch['image'][0:1].to(device)

heatmap = gradcam.generate_heatmap(sample_img_t)

plt.figure(figsize=(6, 6))
plt.title('SignalScope Localized Grad-CAM Heatmap')
plt.imshow(heatmap, cmap='jet')
plt.axis('off')
plt.show()
print('Grad-CAM heatmap generated successfully!')

## 10. Export Calibrated Weights Directly to Google Drive

In [ ]:
import shutil
from pathlib import Path

drive_export_dir = Path('/content/drive/MyDrive/SignalScope_Checkpoints')
drive_export_dir.mkdir(parents=True, exist_ok=True)

search_paths = [
    Path('/content/sih_1/checkpoints/signalscope_final_calibrated.pth'),
    Path('checkpoints/signalscope_final_calibrated.pth'),
    Path('/content/sih_1/checkpoints/signalscope_best_unseen_auc.pth'),
    Path('/content/sih_1/checkpoints/signalscope_best_val_auc.pth'),
    Path('/content/sih_1/checkpoints/signalscope_model_weights.pth')
]

exported = []
for ckpt in search_paths:
    if ckpt.exists():
        dest = drive_export_dir / ckpt.name
        shutil.copy(ckpt, dest)
        exported.append(dest)
        print(f"SUCCESS! Exported: {dest} ({dest.stat().st_size / (1024 * 1024):.2f} MB)")

cal_cfg = Path('/content/sih_1/checkpoints/calibration_config.json')
if cal_cfg.exists():
    shutil.copy(cal_cfg, drive_export_dir / 'calibration_config.json')
    print('Calibration config exported to Google Drive.')

if exported:
    print(f"\nALL SAVED! Model weights successfully stored in Google Drive folder:\n   {drive_export_dir}")
else:
    print('Checkpoint not found. Please ensure Cell 7 (Training) has run and completed.')